In [2]:
import io
import json
import os
import warnings

import geopandas as gpd
import matplotlib
import numpy as np
import pandas as pd
import planetary_computer
import pystac_client
import rasterio
import requests
import stackstac
import torch
import torch.nn as nn
from rasterio.features import rasterize
from rasterio.warp import Resampling, calculate_default_transform, reproject, transform_bounds
from scipy.stats import pearsonr, rankdata, spearmanr
from sklearn.metrics import (average_precision_score, confusion_matrix, f1_score,
                             jaccard_score, precision_recall_curve)

matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib import font_manager
from matplotlib.colors import BoundaryNorm, ListedColormap, to_rgb
from matplotlib.image import imsave
from matplotlib.patches import Patch

In [3]:
warnings.filterwarnings("ignore")

ROOT = "/Users/cyberhbliu/Desktop/PERSONAL/2026portfolio/urban_decay_and_sprawl"
os.chdir(ROOT)
WEB = os.path.join("web", "data")
for folder in ("outputs", "cache", "figures", os.path.join(WEB, "overlays")):
    os.makedirs(folder, exist_ok=True)
print("writing to %s" % ROOT)

TIGER_PLACE = "https://www2.census.gov/geo/tiger/TIGER2024/PLACE/tl_2024_%s_place.zip"
TIGER_TRACT = "https://www2.census.gov/geo/tiger/TIGER2024/TRACT/tl_2024_%s_tract.zip"
TIGER_ROADS = "https://www2.census.gov/geo/tiger/TIGER2024/ROADS/tl_2024_%s_roads.zip"
STAC = "https://planetarycomputer.microsoft.com/api/stac/v1"
WMS = "https://dmsdata.cr.usgs.gov/geoserver/mrlc_Land-Cover-Native_conus_year_data/wms"
WMS_LAYER = "Land-Cover-Native_conus_year_data"
CARTO = "https://phl.carto.com/api/v2/sql"
VPI = ("https://hub.arcgis.com/api/v3/datasets/f7ed68293c5e40d58f1de9c8435c3e84_0"
       "/downloads/data?format=geojson&spatialRefId=4326&where=1%3D1")

CITIES = {"Philadelphia": ("42", "4260000"),
          "Detroit": ("26", "2622000"),
          "Atlanta": ("13", "1304000")}
COUNTY = {"Philadelphia": "42101", "Detroit": "26163", "Atlanta": "13121"}
HOME = "Philadelphia"
T0, T1 = 2015, 2025
YEARS = list(range(T0, T1 + 1))
EARLY, LATE = [2015, 2016, 2017], [2023, 2024, 2025]

BANDS = ["blue", "green", "red", "nir08", "swir16", "swir22", "lwir11"]
NDVI0, NDBI0, NDVI1, NDBI1 = 7, 8, 16, 17
INTENSITY = {21: 1, 22: 2, 23: 3, 24: 4}
WATER, WETLAND = (11, 12), (90, 95)
ROAD_KEEP = ("S1100", "S1200", "S1400")
NLCD_RGB = {(70, 107, 159): 11, (209, 222, 248): 12, (222, 197, 197): 21,
            (217, 146, 130): 22, (235, 0, 0): 23, (171, 0, 0): 24,
            (179, 172, 159): 31, (104, 171, 95): 41, (28, 95, 44): 42,
            (181, 197, 143): 43, (204, 184, 121): 52, (223, 223, 194): 71,
            (220, 217, 57): 81, (171, 108, 40): 82, (184, 217, 235): 90,
            (108, 159, 184): 95}

PIXEL_M, TILE = 30, 1800
PATCH, STRIDE, BLOCK_PX = 128, 64, 256
EPOCHS, BATCH, SEED = 40, 16, 650
CELL_SIZES, PRIMARY_CELL = [90, 150, 300, 600, 1200, 2400], 300
SLIVER_M2 = 10000

writing to /Users/cyberhbliu/Desktop/PERSONAL/2026portfolio/urban_decay_and_sprawl


In [4]:
PAPER, INK, MUTE, WHITE = "#f1f2e0", "#381f04", "#5a544c", "#ffffff"
SAND, GOLD, LIME, UMBER, STONE = "#dfd78b", "#dfad10", "#bad012", "#68542d", "#5a544c"
FURNITURE = "#c9c4ab"
DOT = TRACT_LINE = FURNITURE
SEQUENTIAL = [WHITE, SAND, GOLD, UMBER, INK]
CLASS_COLOR = {"new development": LIME, "decay": GOLD}
CITY_COLOR = {"Philadelphia": SAND, "Detroit": "#9ab0a6", "Atlanta": "#acd2d6"}
MODEL_FILL = {"spectral null": WHITE, "u-net global": GOLD, "u-net per-city": UMBER}

STROKE, DOT_SPACING_M, DOT_SIZE = 2.6, 420, 1.5
SOURCE_NOTE = ("Landsat Collection 2 surface reflectance via Microsoft Planetary Computer. "
               "Land cover from USGS Annual NLCD Collection 1.2. Municipal and census tract "
               "boundaries from US Census TIGER/Line 2024. Vacancy, demolition permits and "
               "311 records from the City of Philadelphia. 30 m resolution, 2015 to 2025.")

available = {f.name for f in font_manager.fontManager.ttflist}
FONT = next(f for f in ["Archivo Black", "Anton", "Helvetica Neue", "Arial Black",
                        "Helvetica", "Arial", "DejaVu Sans"] if f in available)
plt.rcParams.update({
    "figure.facecolor": PAPER, "axes.facecolor": PAPER, "savefig.facecolor": PAPER,
    "font.family": FONT, "text.color": INK, "axes.edgecolor": INK,
    "axes.linewidth": STROKE, "axes.labelcolor": INK, "axes.labelsize": 12,
    "axes.labelweight": "bold", "xtick.color": INK, "ytick.color": INK,
    "xtick.labelsize": 11, "ytick.labelsize": 11, "xtick.major.width": STROKE,
    "ytick.major.width": STROKE, "legend.frameon": False,
    "figure.dpi": 300, "savefig.dpi": 300, "savefig.transparent": False,
})

torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [5]:
def cached(name, build):
    path = os.path.join("cache", name + ".npz")
    if os.path.exists(path):
        with np.load(path, allow_pickle=True) as z:
            return {k: z[k] for k in z.files}
    data = build()
    np.savez_compressed(path, **data)
    return data


def wms_cover(bounds, height, width, year):
    minx, _, _, maxy = bounds
    canvas = None
    for r0 in range(0, height, TILE):
        for c0 in range(0, width, TILE):
            h, w = min(TILE, height - r0), min(TILE, width - c0)
            x0, y1 = minx + c0 * PIXEL_M, maxy - r0 * PIXEL_M
            resp = requests.get(WMS, timeout=300, params={
                "service": "WMS", "version": "1.1.1", "request": "GetMap",
                "layers": WMS_LAYER, "styles": "", "srs": "EPSG:5070",
                "bbox": "%f,%f,%f,%f" % (x0, y1 - h * PIXEL_M, x0 + w * PIXEL_M, y1),
                "width": w, "height": h, "format": "image/geotiff",
                "transparent": "false", "time": "%d-01-01T00:00:00.000Z" % year})
            resp.raise_for_status()
            if b"ServiceException" in resp.content[:2000]:
                raise SystemExit(resp.content[:600].decode("utf-8", "ignore"))
            with rasterio.MemoryFile(resp.content) as mem, mem.open() as src:
                block = src.read()
            if canvas is None:
                canvas = np.zeros((block.shape[0], height, width), block.dtype)
            canvas[:, r0:r0 + h, c0:c0 + w] = block[:, :h, :w]
    if canvas.shape[0] == 1:
        return canvas[0].astype(np.uint8)
    out = np.zeros((height, width), np.uint8)
    rgb = canvas[:3].transpose(1, 2, 0)
    for colour, code in NLCD_RGB.items():
        out[np.all(rgb == np.array(colour, canvas.dtype), axis=-1)] = code
    return out


def carto(query, timeout=900):
    resp = requests.get(CARTO, params={"q": query, "format": "csv"}, timeout=timeout)
    if resp.status_code != 200:
        print("      carto HTTP %d  %s" % (resp.status_code, resp.text[:160]))
        return None
    return pd.read_csv(io.StringIO(resp.text))


def frame(fig, title, filename, top=0.80, bottom=0.20, left=0.05, right=0.965,
          wspace=0.10):
    fig.text(0.035, 0.968, title, ha="left", va="top", fontsize=24,
             fontweight="bold", color=INK)
    fig.text(0.035, 0.018, SOURCE_NOTE, ha="left", va="bottom", fontsize=7.4,
             color=MUTE, wrap=True)
    fig.subplots_adjust(top=min(top, 1 - 1.25 / fig.get_figheight()),
                        bottom=bottom, left=left, right=right, wspace=wspace)
    fig.patch.set_alpha(1.0)
    for ax in fig.axes:
        ax.patch.set_alpha(1.0)
    path = os.path.join("figures", filename)
    fig.savefig(path, facecolor=PAPER, edgecolor="none", transparent=False)
    plt.close(fig)
    print("      %s" % path)


def legend(fig, handles, ncol, y=0.885, size=10, loc="upper left"):
    fig.legend(handles=handles, loc=loc, bbox_to_anchor=(0.035, y), ncol=ncol,
               prop={"weight": "bold", "size": size},
               handlelength=1.5, handleheight=1.2)


def popart_map(ax, city, layers):
    inside, extent = DATA[city]["inside"], DATA[city]["extent"]
    minx, maxx, miny, maxy = extent
    ax.imshow(np.where(inside, 0.0, np.nan), extent=extent,
              cmap=ListedColormap([WHITE]), interpolation="nearest", zorder=1)
    gx, gy = np.meshgrid(np.arange(minx + DOT_SPACING_M / 2, maxx, DOT_SPACING_M),
                         np.arange(miny + DOT_SPACING_M / 2, maxy, DOT_SPACING_M))
    h, w = inside.shape
    col = ((gx - minx) / (maxx - minx) * w).astype(int).clip(0, w - 1)
    row = ((maxy - gy) / (maxy - miny) * h).astype(int).clip(0, h - 1)
    hit = inside[row, col]
    ax.scatter(gx[hit], gy[hit], s=DOT_SIZE, c=DOT, linewidths=0, zorder=2)
    TRACTS[city].boundary.plot(ax=ax, color=TRACT_LINE, linewidth=0.4, zorder=3)
    for arr, colour in layers:
        ax.imshow(np.where(arr, 1.0, np.nan), extent=extent,
                  cmap=ListedColormap([colour]), interpolation="nearest", zorder=4)
    BOUNDS[BOUNDS.city == city].boundary.plot(ax=ax, color=INK,
                                              linewidth=STROKE, zorder=5)
    ax.set_axis_off()


def quantile_choropleth(ax, frame_, column, title):
    values = frame_[column].to_numpy()
    positive = values[values > 0]
    cuts = np.quantile(positive, [.2, .4, .6, .8]) if len(positive) > 4 else [1, 2, 3, 4]
    edges = np.unique(np.concatenate([[-1e-9], cuts, [values.max() + 1]]))
    while len(edges) < 6:
        edges = np.append(edges, edges[-1] + 1)
    frame_.plot(ax=ax, column=column, cmap=ListedColormap(SEQUENTIAL),
                norm=BoundaryNorm(edges[:6], 5), edgecolor=MUTE, linewidth=0.3)
    BOUNDS[BOUNDS.city == HOME].boundary.plot(ax=ax, color=INK, linewidth=STROKE)
    ax.set_axis_off()
    ax.set_title(title, loc="left", fontsize=13, fontweight="bold", pad=8)


def overlay_png(mask, extent, colour, name):
    minx, maxx, miny, maxy = extent
    h, w = mask.shape
    dst, dw, dh = calculate_default_transform("EPSG:5070", "EPSG:3857", w, h,
                                              minx, miny, maxx, maxy)
    out = np.zeros((dh, dw), np.uint8)
    reproject(mask.astype(np.uint8), out,
              src_transform=rasterio.transform.from_origin(minx, maxy, PIXEL_M, PIXEL_M),
              src_crs="EPSG:5070", dst_transform=dst, dst_crs="EPSG:3857",
              resampling=Resampling.nearest)
    rgba = np.zeros((dh, dw, 4), np.uint8)
    rgba[..., :3] = np.round(np.array(to_rgb(colour)) * 255)
    rgba[..., 3] = np.where(out > 0, 255, 0)
    imsave(os.path.join(WEB, "overlays", name + ".png"), rgba)
    lon0, lat0, lon1, lat1 = transform_bounds(
        "EPSG:3857", "EPSG:4326", *rasterio.transform.array_bounds(dh, dw, dst))
    return {"file": "data/overlays/%s.png" % name, "bounds": [[lon0, lat1], [lon1, lat0]]}


# 1. fetching NLCD labels over WMS

In [6]:
GRID = {}
for city, (state, place) in CITIES.items():
    shape = gpd.read_file(TIGER_PLACE % state)
    shape = shape[shape["GEOID"] == place].to_crs("EPSG:5070")
    minx, miny, maxx, maxy = shape.total_bounds
    minx, maxy = np.floor(minx / PIXEL_M) * PIXEL_M, np.ceil(maxy / PIXEL_M) * PIXEL_M
    width = int(np.ceil((maxx - minx) / PIXEL_M))
    height = int(np.ceil((maxy - miny) / PIXEL_M))
    maxx, miny = minx + width * PIXEL_M, maxy - height * PIXEL_M
    transform = rasterio.transform.from_origin(minx, maxy, PIXEL_M, PIXEL_M)
    GRID[city] = {
        "shape": shape, "height": height, "width": width, "transform": transform,
        "bounds": (minx, miny, maxx, maxy),
        "inside": rasterize([(g, 1) for g in shape.geometry], out_shape=(height, width),
                            transform=transform, fill=0, dtype="uint8").astype(bool)}
    print("      %-13s %d x %d" % (city, height, width))

BOUNDS = pd.concat([GRID[c]["shape"].assign(city=c) for c in CITIES])
BOUNDS.to_file("outputs/boundaries.geojson", driver="GeoJSON")


def build_cover():
    out = {}
    for city in CITIES:
        g = GRID[city]
        for year in YEARS:
            out["%s|%d" % (city, year)] = wms_cover(g["bounds"], g["height"],
                                                    g["width"], year)
            print("      %-13s %d" % (city, year))
    return out


COVER = cached("cover", build_cover)
RANK = {}
for city in CITIES:
    for year in YEARS:
        cover = COVER["%s|%d" % (city, year)]
        rank = np.zeros(cover.shape, np.int8)
        for code, level in INTENSITY.items():
            rank[cover == code] = level
        RANK[(city, year)] = rank

      Philadelphia  1109 x 818
      Detroit       803 x 1032
      Atlanta       903 x 791


# 2. building imagery and labels

In [7]:
catalog = pystac_client.Client.open(STAC, modifier=planetary_computer.sign_inplace)


def build_imagery():
    out = {}
    for city in CITIES:
        g = GRID[city]
        wgs = g["shape"].to_crs("EPSG:4326").total_bounds
        scenes = []
        for year in (T0, T1):
            items = catalog.search(
                collections=["landsat-c2-l2"], bbox=list(wgs),
                datetime="%d-05-01/%d-09-30" % (year, year),
                query={"eo:cloud_cover": {"lt": 30},
                       "platform": {"in": ["landsat-8", "landsat-9"]}}).item_collection()
            if not len(items):
                raise SystemExit("no Landsat scenes for %s %d" % (city, year))
            cube = stackstac.stack(items, assets=BANDS + ["qa_pixel"], epsg=5070,
                                   resolution=PIXEL_M, bounds=g["bounds"],
                                   chunksize=1024, dtype="float32",
                                   fill_value=np.float32("nan"), rescale=False)
            qa = cube.sel(band="qa_pixel").fillna(0).astype("uint16")
            clear = ((qa & 2) == 0) & ((qa & 8) == 0) & ((qa & 16) == 0)
            stack = cube.sel(band=BANDS).where(clear).median("time").compute().values
            optical = stack[:6] * 0.0000275 - 0.2
            thermal = (stack[6] * 0.00341802 + 149.0 - 273.15) / 50.0
            nd = lambda a, b: (a - b) / (a + b + 1e-6)
            scenes.append(np.concatenate([
                optical, thermal[None],
                nd(optical[3], optical[2])[None],
                nd(optical[4], optical[3])[None]]).astype(np.float32))
            print("      %-13s %d  %d scenes" % (city, year, len(items)))
        image = np.nan_to_num(np.concatenate(scenes), nan=0.0)
        target = (image.shape[0], g["height"], g["width"])
        if image.shape != target:
            pad = np.zeros(target, np.float32)
            h, w = min(target[1], image.shape[1]), min(target[2], image.shape[2])
            pad[:, :h, :w] = image[:, :h, :w]
            image = pad
        out[city] = image
    return out


IMAGERY = cached("imagery", build_imagery)
DATA = {}
for city in CITIES:
    g = GRID[city]
    inside, height, width = g["inside"], g["height"], g["width"]
    early = np.median(np.stack([RANK[(city, y)] for y in EARLY]), axis=0)
    late = np.median(np.stack([RANK[(city, y)] for y in LATE]), axis=0)
    cov0 = COVER["%s|%d" % (city, T0)]
    convertible = (early == 0) & ~np.isin(cov0, WATER) & ~np.isin(cov0, WETLAND)
    developed = early >= 1
    label = np.zeros((height, width), np.int64)
    label[convertible & (late >= 1)] = 1
    label[developed & (late < early)] = 2
    label[~inside] = 0
    rows, cols = np.mgrid[0:height, 0:width]
    minx, miny, maxx, maxy = g["bounds"]
    DATA[city] = {"image": IMAGERY[city], "label": label, "inside": inside,
                  "developed": developed & inside, "convertible": convertible & inside,
                  "holdout": ((rows // BLOCK_PX) + (cols // BLOCK_PX)) % 2 == 1,
                  "transform": g["transform"], "extent": (minx, maxx, miny, maxy)}
    print("      %-13s new development %d  decay %d"
          % (city, (label == 1).sum(), (label == 2).sum()))


      Philadelphia  new development 1426  decay 4424
      Detroit       new development 103  decay 11211
      Atlanta       new development 3010  decay 6124


# 3. training the U-Net

In [8]:
class UNet(nn.Module):
    def __init__(self, ch, out):
        super().__init__()
        block = lambda i, o: nn.Sequential(
            nn.Conv2d(i, o, 3, padding=1), nn.BatchNorm2d(o), nn.ReLU(True),
            nn.Conv2d(o, o, 3, padding=1), nn.BatchNorm2d(o), nn.ReLU(True))
        self.e1, self.e2, self.e3 = block(ch, 32), block(32, 64), block(64, 128)
        self.pool = nn.MaxPool2d(2)
        self.u2, self.d2 = nn.ConvTranspose2d(128, 64, 2, 2), block(128, 64)
        self.u1, self.d1 = nn.ConvTranspose2d(64, 32, 2, 2), block(64, 32)
        self.head = nn.Conv2d(32, out, 1)

    def forward(self, x):
        a = self.e1(x)
        b = self.e2(self.pool(a))
        c = self.e3(self.pool(b))
        return self.head(self.d1(torch.cat(
            [self.u1(self.d2(torch.cat([self.u2(c), b], 1))), a], 1)))


home = DATA[HOME]
patches = [(home["image"][:, r:r + PATCH, c:c + PATCH],
            home["label"][r:r + PATCH, c:c + PATCH],
            home["inside"][r:r + PATCH, c:c + PATCH])
           for r in range(0, home["label"].shape[0] - PATCH + 1, STRIDE)
           for c in range(0, home["label"].shape[1] - PATCH + 1, STRIDE)
           if home["inside"][r:r + PATCH, c:c + PATCH].mean() >= 0.5
           and not home["holdout"][r:r + PATCH, c:c + PATCH].any()]

stackX = np.stack([p[0] for p in patches])
MU, SD = stackX.mean(axis=(0, 2, 3)), stackX.std(axis=(0, 2, 3)) + 1e-6
Xt = torch.from_numpy((stackX - MU[None, :, None, None]) / SD[None, :, None, None]).float()
yt = torch.from_numpy(np.stack([p[1] for p in patches]))
mt = torch.from_numpy(np.stack([p[2] for p in patches]).astype(np.float32))
print("      %d patches, %d channels" % (Xt.shape[0], Xt.shape[1]))

counts = np.array([(yt.numpy() == k).sum() for k in range(3)], np.float64) + 1
raw = (counts.sum() / counts) ** 0.5
weights = torch.tensor(raw / raw[0], dtype=torch.float32, device=DEVICE)
print("      class weights %s" % np.round(weights.cpu().numpy(), 2))

net = UNet(Xt.shape[1], 3).to(DEVICE)
if os.path.exists("cache/unet.pt"):
    net.load_state_dict(torch.load("cache/unet.pt", map_location=DEVICE))
    print("      loaded cached weights")
else:
    opt = torch.optim.Adam(net.parameters(), 1e-3)
    ce = nn.CrossEntropyLoss(weight=weights, reduction="none")
    for epoch in range(EPOCHS):
        net.train()
        order = torch.randperm(Xt.shape[0])
        total = 0.0
        for s in range(0, len(order), BATCH):
            i = order[s:s + BATCH]
            xb, yb, mb = Xt[i].to(DEVICE), yt[i].to(DEVICE), mt[i].to(DEVICE)
            opt.zero_grad()
            logits = net(xb)
            loss = (ce(logits, yb) * mb).sum() / mb.sum().clamp(min=1)
            prob = torch.softmax(logits, 1)
            for k in (1, 2):
                p, t = prob[:, k] * mb, (yb == k).float() * mb
                tp, fp, fn = (p * t).sum(), (p * (1 - t)).sum(), ((1 - p) * t).sum()
                loss = loss + (1 - (tp + 1) / (tp + 0.3 * fp + 0.7 * fn + 1))
            loss.backward()
            opt.step()
            total += float(loss) * len(i)
        if epoch % 10 == 9:
            print("      epoch %2d  loss %.4f" % (epoch + 1, total / Xt.shape[0]))
    torch.save(net.state_dict(), "cache/unet.pt")



      28 patches, 18 channels
      class weights [ 1.   18.58 10.16]
      loaded cached weights


# 4. prediction, transfer and spectral null

In [9]:
net.eval()
PRED, PROB, metrics = {}, {}, []

for city in CITIES:
    d = DATA[city]
    height, width = d["label"].shape
    keep = (d["inside"] & d["holdout"]) if city == HOME else d["inside"]
    kind = "spatial holdout" if city == HOME else "transfer"

    for scheme in ("global", "per-city"):
        if scheme == "global":
            mu, sd = MU, SD
        else:
            flat = d["image"].reshape(d["image"].shape[0], -1)
            mu, sd = flat.mean(1), flat.std(1) + 1e-6
        norm = ((d["image"] - mu[:, None, None]) / sd[:, None, None]).astype(np.float32)

        acc = np.zeros((3, height, width), np.float32)
        cnt = np.zeros((height, width), np.float32)
        rows = sorted({*range(0, max(height - PATCH, 0) + 1, STRIDE), max(height - PATCH, 0)})
        cols = sorted({*range(0, max(width - PATCH, 0) + 1, STRIDE), max(width - PATCH, 0)})
        with torch.no_grad():
            for r in rows:
                tiles = np.stack([norm[:, r:r + PATCH, c:c + PATCH] for c in cols])
                out = torch.softmax(net(torch.from_numpy(tiles).to(DEVICE)), 1).cpu().numpy()
                for k, c in enumerate(cols):
                    acc[:, r:r + PATCH, c:c + PATCH] += out[k]
                    cnt[r:r + PATCH, c:c + PATCH] += 1
        prob = acc / np.maximum(cnt, 1)
        pred = prob.argmax(0)
        pred[~d["inside"]] = 0
        if scheme == "per-city":
            PRED[city], PROB[city] = pred, prob
            np.savetxt("outputs/confusion_%s.csv" % city.lower(),
                       confusion_matrix(d["label"][keep], pred[keep], labels=[0, 1, 2]),
                       fmt="%d", delimiter=",")

        truth, guess = d["label"][keep], pred[keep]
        f1 = f1_score(truth, guess, labels=[0, 1, 2], average=None, zero_division=0)
        iou = jaccard_score(truth, guess, labels=[0, 1, 2], average=None, zero_division=0)
        row = {"city": city, "kind": kind, "model": "u-net %s" % scheme}
        for k, name in ((1, "new development"), (2, "decay")):
            binary = (truth == k).astype(int)
            row["f1_" + name] = f1[k]
            row["iou_" + name] = iou[k]
            row["ap_" + name] = average_precision_score(binary, prob[k][keep]) if binary.sum() else np.nan
            row["prevalence_" + name] = binary.mean()
            row["predicted_km2_" + name] = (guess == k).sum() * 0.0009
            row["observed_km2_" + name] = binary.sum() * 0.0009
        metrics.append(row)
        print("      %-13s %-8s  AP new %.3f  AP decay %.3f"
              % (city, scheme, row["ap_new development"], row["ap_decay"]))

    row = {"city": city, "kind": kind, "model": "spectral null"}
    for k, name, score, domain in (
            (1, "new development", d["image"][NDBI1] - d["image"][NDBI0], d["convertible"]),
            (2, "decay", d["image"][NDVI1] - d["image"][NDVI0], d["developed"])):
        mask = keep & domain
        binary = (d["label"][mask] == k).astype(int)
        row["ap_" + name] = average_precision_score(binary, score[mask]) if binary.sum() else np.nan
    metrics.append(row)
    print("      %-13s null      AP new %.3f  AP decay %.3f"
          % (city, row["ap_new development"], row["ap_decay"]))

METRICS = pd.DataFrame(metrics)
METRICS.to_csv("outputs/metrics.csv", index=False)

      Philadelphia  global    AP new 0.153  AP decay 0.061
      Philadelphia  per-city  AP new 0.159  AP decay 0.052
      Philadelphia  null      AP new 0.407  AP decay 0.026
      Detroit       global    AP new 0.005  AP decay 0.030
      Detroit       per-city  AP new 0.002  AP decay 0.034
      Detroit       null      AP new 0.078  AP decay 0.022
      Atlanta       global    AP new 0.179  AP decay 0.018
      Atlanta       per-city  AP new 0.107  AP decay 0.017
      Atlanta       null      AP new 0.444  AP decay 0.019


# 5. Philadelphia municipal records

In [10]:
def build_records():
    out = {}
    names = carto("SELECT service_name FROM public_cases_fc WHERE requested_datetime >= "
                  "'%d-01-01' AND requested_datetime < '%d-01-01' AND (service_name ILIKE "
                  "'%%vacant%%' OR service_name ILIKE '%%abandon%%' OR service_name ILIKE "
                  "'%%dangerous%%') GROUP BY service_name" % (T1 - 1, T1))
    if names is not None:
        quoted = ", ".join("'" + x.replace("'", "''") + "'"
                           for x in names["service_name"].dropna())
        parts = [carto("SELECT lat, lon FROM public_cases_fc WHERE lat IS NOT NULL AND "
                       "service_name IN (%s) AND requested_datetime >= '%d-01-01' AND "
                       "requested_datetime < '%d-01-01'" % (quoted, y, y + 1))
                 for y in YEARS]
        parts = [p for p in parts if p is not None]
        if parts:
            out["311 vacancy"] = pd.concat(parts, ignore_index=True) \
                .dropna()[["lat", "lon"]].to_numpy()

    kinds = carto("SELECT permitdescription FROM permits WHERE permitissuedate >= "
                  "'%d-01-01' AND permitdescription ILIKE '%%demolition%%' "
                  "GROUP BY permitdescription" % T0)
    if kinds is not None:
        quoted = ", ".join("'" + x.replace("'", "''") + "'"
                           for x in kinds["permitdescription"].dropna())
        got = carto("SELECT ST_Y(the_geom) AS lat, ST_X(the_geom) AS lon FROM permits "
                    "WHERE the_geom IS NOT NULL AND permitdescription IN (%s) AND "
                    "permitissuedate >= '%d-01-01' AND permitissuedate < '%d-01-01'"
                    % (quoted, T0, T1 + 1))
        if got is not None:
            out["demolition permits"] = got.dropna()[["lat", "lon"]].to_numpy()

    resp = requests.get(VPI, timeout=900)
    if resp.status_code == 200:
        pts = gpd.read_file(io.BytesIO(resp.content)) \
            .set_crs("EPSG:4326", allow_override=True).geometry.representative_point()
        out["vacant buildings"] = np.column_stack([pts.y.values, pts.x.values])
    return {k.replace(" ", "_"): v for k, v in out.items()}


RECORDS = {k.replace("_", " "): v for k, v in cached("records", build_records).items()}
for name, xy in RECORDS.items():
    print("      %-20s %d records" % (name, len(xy)))

      311 vacancy          298608 records
      demolition permits   11209 records
      vacant buildings     8773 records


# 6. scale sensitivity and partial correlation

In [11]:
d = DATA[HOME]
inverse = ~d["transform"]
projected = {}
for name, xy in RECORDS.items():
    pts = gpd.GeoSeries(gpd.points_from_xy(xy[:, 1], xy[:, 0]),
                        crs="EPSG:4326").to_crs("EPSG:5070")
    cols, rows = inverse * (pts.x.values, pts.y.values)
    projected[name] = np.column_stack([rows, cols])

height, width = d["label"].shape
validation = []
for cell in CELL_SIZES:
    factor = cell // PIXEL_M
    rn, cn = height // factor, width // factor
    crop = (slice(0, rn * factor), slice(0, cn * factor))
    agg = {k: a[crop].reshape(rn, factor, cn, factor).sum(axis=(1, 3))
           for k, a in [("area", d["inside"].astype(np.float32)),
                        ("built", d["developed"].astype(np.float32)),
                        ("NLCD label", (d["label"] == 2).astype(np.float32)),
                        ("U-Net detection", (PRED[HOME] == 2).astype(np.float32))]}
    for name, rc in projected.items():
        counts = np.zeros((rn, cn), np.float32)
        r, c = (rc[:, 0] // factor).astype(int), (rc[:, 1] // factor).astype(int)
        ok = (r >= 0) & (r < rn) & (c >= 0) & (c < cn)
        np.add.at(counts, (r[ok], c[ok]), 1)
        agg[name] = counts

    keep = agg["area"] >= 0.5 * factor * factor
    control = rankdata(agg["built"][keep])
    for name in projected:
        y_rank = rankdata(agg[name][keep])
        ry = y_rank - np.polyval(np.polyfit(control, y_rank, 1), control)
        for source in ("NLCD label", "U-Net detection"):
            x_rank = rankdata(agg[source][keep])
            rx = x_rank - np.polyval(np.polyfit(control, x_rank, 1), control)
            rho, p = spearmanr(x_rank, y_rank)
            prho, pp = pearsonr(rx, ry)
            validation.append({"cell_m": cell, "n_cells": int(keep.sum()),
                               "record": name, "decay from": source, "spearman": rho,
                               "p": p, "partial_spearman": prho, "partial_p": pp})

VALIDATION = pd.DataFrame(validation)
VALIDATION.to_csv("outputs/validation.csv", index=False)
for _, r in VALIDATION[VALIDATION.cell_m == PRIMARY_CELL].iterrows():
    print("      %-19s vs %-16s rho %+.3f  partial %+.3f"
          % (r["record"], r["decay from"], r["spearman"], r["partial_spearman"]))

      311 vacancy         vs NLCD label       rho -0.233  partial -0.208
      311 vacancy         vs U-Net detection  rho -0.269  partial -0.220
      demolition permits  vs NLCD label       rho -0.065  partial -0.018
      demolition permits  vs U-Net detection  rho -0.151  partial -0.092
      vacant buildings    vs NLCD label       rho -0.141  partial -0.102
      vacant buildings    vs U-Net detection  rho -0.185  partial -0.130


# 7. census tracts

In [12]:
TRACTS = {}
for city, (state, _) in CITIES.items():
    part = gpd.read_file(TIGER_TRACT % state).to_crs("EPSG:5070")
    geom = BOUNDS[BOUNDS.city == city].geometry.union_all().buffer(0)
    hit = part[part.intersects(geom)].copy()
    hit["geometry"] = hit.geometry.buffer(0).intersection(geom)
    hit = hit[~hit.geometry.is_empty].explode(index_parts=False)
    hit = hit[(hit.geom_type == "Polygon") & (hit.geometry.area >= SLIVER_M2)]
    TRACTS[city] = hit.dissolve(by="GEOID", as_index=False).reset_index(drop=True)
    print("      %-13s %d tracts" % (city, len(TRACTS[city])))

tracts = TRACTS[HOME].copy()
index = rasterize([(g, i + 1) for i, g in enumerate(tracts.geometry)],
                  out_shape=d["inside"].shape, transform=d["transform"],
                  fill=0, dtype="int32")
flat, n = index.ravel(), len(tracts) + 1
tracts["land_km2"] = np.bincount(flat, weights=d["inside"].ravel().astype(float),
                                 minlength=n)[1:] * 0.0009
tracts["NLCD label"] = np.bincount(flat, weights=(d["label"] == 2).ravel().astype(float),
                                   minlength=n)[1:] * 0.0009
tracts["U-Net detection"] = np.bincount(flat, weights=(PRED[HOME] == 2).ravel().astype(float),
                                        minlength=n)[1:] * 0.0009
for name, rc in projected.items():
    r, c = rc[:, 0].astype(int), rc[:, 1].astype(int)
    ok = (r >= 0) & (r < d["inside"].shape[0]) & (c >= 0) & (c < d["inside"].shape[1])
    tracts[name] = np.bincount(index[r[ok], c[ok]], minlength=n)[1:].astype(float)

tracts = tracts[tracts["land_km2"] > 0.02].copy()
for col in ["U-Net detection", "NLCD label"] + list(projected):
    tracts[col + " density"] = tracts[col] / tracts["land_km2"]
DENSITY = [c for c in tracts.columns if c.endswith(" density")]
tracts.drop(columns="geometry").to_csv("outputs/tract_summary.csv", index=False)
tracts.to_file("outputs/philadelphia_tracts.geojson", driver="GeoJSON")
print("      %d tracts with land area" % len(tracts))

      Philadelphia  408 tracts
      Detroit       276 tracts
      Atlanta       192 tracts
      408 tracts with land area


# 8. annual trajectory

In [13]:
rows = []
for city in CITIES:
    inside = DATA[city]["inside"]
    base_mask = base_area = None
    for year in YEARS:
        developed = (RANK[(city, year)] >= 1) & inside
        area = developed.sum() * 0.0009
        if base_mask is None:
            base_mask, base_area = developed.copy(), max(area, 1e-9)
        rows.append({"city": city, "year": year, "developed_km2": area,
                     "developed_index": 100.0 * area / base_area,
                     "mean_intensity": float(RANK[(city, year)][base_mask].mean())})
TRAJECTORY = pd.DataFrame(rows)
TRAJECTORY["intensity_change"] = TRAJECTORY.groupby("city")["mean_intensity"] \
    .transform(lambda s: s - s.iloc[0])
TRAJECTORY.to_csv("outputs/trajectory.csv", index=False)

REGIME = METRICS[METRICS.model == "u-net per-city"].set_index("city").reindex(CITIES)
REGIME[["predicted_km2_new development", "predicted_km2_decay"]] \
    .assign(regime_index=lambda t: t.iloc[:, 0] / t.iloc[:, 1].clip(lower=1e-6)) \
    .to_csv("outputs/development_regime.csv")

np.savez_compressed("outputs/figure_data.npz", cities=np.array(list(CITIES)), **{
    "%s__%s" % (city.lower(), field): value
    for city in CITIES
    for field, value in [("label", DATA[city]["label"].astype(np.int8)),
                         ("pred", PRED[city].astype(np.int8)),
                         ("prob_decay", PROB[city][2].astype(np.float16)),
                         ("prob_new", PROB[city][1].astype(np.float16)),
                         ("inside", DATA[city]["inside"]),
                         ("holdout", DATA[city]["holdout"]),
                         ("extent", np.array(DATA[city]["extent"]))]})

# 9. viz

In [14]:
CLASS_LEGEND = [Patch(facecolor=CLASS_COLOR[c], edgecolor=INK, linewidth=1.6,
                      label=c.upper()) for c in ("new development", "decay")]
CITY_LEGEND = [plt.Line2D([], [], color=CITY_COLOR[c], linewidth=3.4, marker="o",
                          markersize=8, markeredgecolor=INK, markeredgewidth=1.4,
                          label=c.upper()) for c in CITIES]
RAMP_LEGEND = [Patch(facecolor=SEQUENTIAL[i], edgecolor=INK, linewidth=1.3,
                     label=["LOWEST FIFTH", "SECOND", "THIRD", "FOURTH",
                            "HIGHEST FIFTH"][i]) for i in range(5)]

fig, axes = plt.subplots(1, 3, figsize=(14, 9))
for ax, city in zip(axes, CITIES):
    popart_map(ax, city, [(PRED[city] == 1, CLASS_COLOR["new development"]),
                          (PRED[city] == 2, CLASS_COLOR["decay"])])
    ax.set_title(city.upper(), loc="left", fontsize=14, fontweight="bold", pad=8)
legend(fig, CLASS_LEGEND + [plt.Line2D([], [], color=TRACT_LINE, linewidth=1.2,
                                       label="CENSUS TRACT")], 3, y=0.075,
       loc="lower left")
frame(fig, "DETECTED NEW DEVELOPMENT AND DECAY, 2015 TO 2025",
      "fig01_detected_change.png", top=0.88, bottom=0.14)

fig, axes = plt.subplots(1, 2, figsize=(14, 7))
for ax, col, ylabel in zip(axes, ["developed_index", "intensity_change"],
                           ["DEVELOPED AREA, 2015 = 100",
                            "CHANGE IN MEAN INTENSITY SINCE 2015"]):
    for city in CITIES:
        part = TRAJECTORY[TRAJECTORY.city == city].sort_values("year")
        ax.plot(part.year, part[col], marker="o", markersize=7, linewidth=3.0,
                color=CITY_COLOR[city], markeredgecolor=INK, markeredgewidth=1.4)
    if col == "intensity_change":
        ax.axhline(0, color=INK, linewidth=STROKE)
    ax.set_xlabel("YEAR")
    ax.set_ylabel(ylabel)
    ax.set_xticks(YEARS[::2])
    ax.spines[["top", "right"]].set_visible(False)
legend(fig, CITY_LEGEND, 3)
frame(fig, "ANNUAL DEVELOPED AREA AND INTENSITY", "fig02_trajectory.png",
      top=0.775, bottom=0.23, left=0.095, wspace=0.32)

fig, axes = plt.subplots(1, 2, figsize=(14, 8))
for ax, arr, name in zip(axes, [DATA[HOME]["label"], PRED[HOME]], ["OBSERVED", "DETECTED"]):
    popart_map(ax, HOME, [((arr == 1) & DATA[HOME]["holdout"], CLASS_COLOR["new development"]),
                          ((arr == 2) & DATA[HOME]["holdout"], CLASS_COLOR["decay"])])
    ax.set_title(name, loc="left", fontsize=14, fontweight="bold", pad=8)
legend(fig, CLASS_LEGEND, 2, y=0.07, loc="lower left")
frame(fig, "OBSERVED VERSUS DETECTED CHANGE ON HELD-OUT BLOCKS",
      "fig03_observed_vs_detected.png", top=0.88, bottom=0.13)

order = ["spectral null", "u-net global", "u-net per-city"]
fig, ax = plt.subplots(figsize=(14, 6))
y = np.arange(len(CITIES))[::-1]
h = 0.8 / len(order)
for i, model in enumerate(order):
    values = [METRICS[(METRICS.city == c) & (METRICS.model == model)]["ap_decay"].iloc[0]
              for c in CITIES]
    pos = y + (len(order) - 1 - 2 * i) * h / 2
    ax.barh(pos, values, h * 0.9, color=MODEL_FILL[model], edgecolor=INK, linewidth=1)
    for yy, v in zip(pos, values):
        ax.text(v + 0.002, yy, "%.3f" % v, va="center", fontsize=10, fontweight="bold")
ax.set_yticks(y)
ax.set_yticklabels([c.upper() for c in CITIES], fontweight="bold")
ax.set_xlabel("AVERAGE PRECISION, DECAY CLASS")
ax.spines[["top", "right"]].set_visible(False)
legend(fig, [Patch(facecolor=MODEL_FILL[m], edgecolor=INK, linewidth=1, label=m.upper())
             for m in order], 3)
frame(fig, "DECAY DETECTION AGAINST A SPECTRAL BASELINE",
      "fig04_detection_performance.png", top=0.775, bottom=0.20, left=0.155)

PR_CURVES = []
fig, axes = plt.subplots(1, 2, figsize=(14, 7))
for ax, k, name in zip(axes, (1, 2), ("new development", "decay")):
    for city in CITIES:
        dd = DATA[city]
        mask = (dd["inside"] & dd["holdout"]) if city == HOME else dd["inside"]
        truth = (dd["label"][mask] == k).astype(int)
        if not truth.sum():
            continue
        precision, recall, _ = precision_recall_curve(truth, PROB[city][k][mask])
        step = max(1, len(recall) // 240)
        PR_CURVES.append({"city": city, "class": name, "prevalence": float(truth.mean()),
                          "points": [[float(a), float(b)]
                                     for a, b in zip(recall[::step], precision[::step])]})
        ax.plot(recall, precision, linewidth=3.0, color=CITY_COLOR[city])
        ax.axhline(truth.mean(), color=CITY_COLOR[city], linewidth=1.3,
                   linestyle=(0, (2, 2)))
    ax.set_title(name.upper(), loc="left", fontsize=13, fontweight="bold", pad=8)
    ax.set_xlabel("RECALL")
    ax.set_ylabel("PRECISION")
    ax.set_xlim(0, 1)
    ax.spines[["top", "right"]].set_visible(False)
legend(fig, CITY_LEGEND, 3, y=0.075, loc="lower left")
frame(fig, "PRECISION AND RECALL AGAINST CLASS PREVALENCE",
      "fig05_precision_recall.png", top=0.87, bottom=0.20, left=0.085, wspace=0.24)

fig, ax = plt.subplots(figsize=(14, 6))
y = np.arange(len(CITIES))[::-1]
top_value = REGIME[["predicted_km2_new development", "predicted_km2_decay"]].to_numpy().max()
for offset, name in ((0.19, "new development"), (-0.19, "decay")):
    values = REGIME["predicted_km2_" + name].to_numpy()
    ax.barh(y + offset, values, 0.34, color=CLASS_COLOR[name], edgecolor=INK, linewidth=1)
    for yy, v in zip(y + offset, values):
        ax.text(v + top_value * 0.012, yy, "%.2f km\u00b2" % v, va="center",
                fontsize=10, fontweight="bold")
ax.set_yticks(y)
ax.set_yticklabels([c.upper() for c in CITIES], fontweight="bold")
ax.set_xlabel("AREA DETECTED, SQUARE KILOMETRES")
ax.set_xlim(0, top_value * 1.2)
ax.spines[["top", "right"]].set_visible(False)
legend(fig, CLASS_LEGEND, 2)
frame(fig, "DETECTED CHANGE AREA BY CITY AND CLASS", "fig06_change_area.png",
      top=0.775, bottom=0.20, left=0.155)

fig, axes = plt.subplots(1, 2, figsize=(14, 8))
for ax, col in zip(axes, ["U-Net detection", "NLCD label"]):
    quantile_choropleth(ax, tracts, col + " density", col.upper())
legend(fig, RAMP_LEGEND, 5, y=0.07, size=9.5, loc="lower left")
frame(fig, "SATELLITE DECAY BY CENSUS TRACT, PHILADELPHIA",
      "fig07_satellite_by_tract.png", top=0.88, bottom=0.14)

fig, axes = plt.subplots(1, len(projected), figsize=(5.2 * len(projected), 8.4))
for ax, name in zip(np.atleast_1d(axes), projected):
    quantile_choropleth(ax, tracts, name + " density", name.upper())
legend(fig, RAMP_LEGEND, 5, y=0.07, size=9.5, loc="lower left")
frame(fig, "MUNICIPAL VACANCY RECORDS BY CENSUS TRACT, PHILADELPHIA",
      "fig08_records_by_tract.png", top=0.88, bottom=0.14)

fig, axes = plt.subplots(1, len(projected), figsize=(5.2 * len(projected), 6.8))
for i, (ax, name) in enumerate(zip(np.atleast_1d(axes), projected)):
    x = tracts["U-Net detection density"].to_numpy()
    z = tracts[name + " density"].to_numpy()
    rho, p = spearmanr(x, z)
    ax.scatter(x, z, s=10, color=STONE, edgecolor=INK, linewidths=0.3, zorder=3)
    slope, intercept = np.polyfit(x, z, 1)
    grid = np.linspace(x.min(), x.max(), 50)
    ax.plot(grid, slope * grid + intercept, color=INK, linewidth=1, zorder=4)
    ax.set_ylim(0, z.max() * 1.06)
    ax.text(0.97, 0.05, "SPEARMAN %+.3f\np %.1e" % (rho, p), transform=ax.transAxes,
            ha="right", va="bottom", fontsize=10, fontweight="bold", color=INK)
    ax.set_title(name.upper(), loc="left", fontsize=13, fontweight="bold", pad=8)
    ax.set_xlabel("DETECTED DECAY, SHARE OF TRACT LAND")
    if i == 0:
        ax.set_ylabel("RECORDS PER SQUARE KILOMETRE")
    ax.spines[["top", "right"]].set_visible(False)
frame(fig, "DETECTED DECAY AGAINST RECORDED VACANCY, n = %d TRACTS" % len(tracts),
      "fig09_tract_scatter.png", top=0.84, bottom=0.17, left=0.07, wspace=0.26)

subset = VALIDATION[VALIDATION["decay from"] == "U-Net detection"]
fig, axes = plt.subplots(1, len(projected), figsize=(5.2 * len(projected), 6.8),
                         sharey=True)
for i, (ax, name) in enumerate(zip(np.atleast_1d(axes), projected)):
    part = subset[subset.record == name].sort_values("cell_m")
    ax.plot(part.cell_m, part.spearman, marker="o", markersize=9, linewidth=3.2,
            color=CLASS_COLOR["decay"], markeredgecolor=INK, markeredgewidth=1.6)
    ax.plot(part.cell_m, part.partial_spearman, marker="s", markersize=8, linewidth=3.2,
            linestyle=(0, (3, 2)), color=INK, markeredgecolor=INK, markeredgewidth=1.6)
    ax.axhline(0, color=INK, linewidth=STROKE)
    ax.set_xscale("log")
    ax.set_xticks(CELL_SIZES)
    ax.set_xticklabels([str(s) for s in CELL_SIZES], fontweight="bold", fontsize=9)
    ax.set_title(name.upper(), loc="left", fontsize=13, fontweight="bold", pad=8)
    ax.set_xlabel("CELL SIZE, METRES")
    if i == 0:
        ax.set_ylabel("SPEARMAN WITH DETECTED DECAY")
    ax.spines[["top", "right"]].set_visible(False)
legend(fig, [plt.Line2D([], [], color=CLASS_COLOR["decay"], linewidth=3.2, marker="o",
                        markersize=8, markeredgecolor=INK, label="BIVARIATE"),
             plt.Line2D([], [], color=INK, linewidth=3.2, linestyle=(0, (3, 2)),
                        marker="s", markersize=7, label="CONTROLLING FOR BUILT AREA")], 2)
frame(fig, "CORRELATION AGAINST AGGREGATION SCALE", "fig10_scale_sensitivity.png",
      top=0.775, bottom=0.17, left=0.085, wspace=0.16)

      figures/fig01_detected_change.png
      figures/fig02_trajectory.png
      figures/fig03_observed_vs_detected.png
      figures/fig04_detection_performance.png
      figures/fig05_precision_recall.png
      figures/fig06_change_area.png
      figures/fig07_satellite_by_tract.png
      figures/fig08_records_by_tract.png
      figures/fig09_tract_scatter.png
      figures/fig10_scale_sensitivity.png


# 10. web export

In [19]:
OVERLAYS = {}
for city in CITIES:
    dd = DATA[city]
    views = [("detected", PRED[city], None)]
    if city == HOME:
        views += [("observed", dd["label"], dd["holdout"]),
                  ("detectedholdout", PRED[city], dd["holdout"])]
    for view, arr, gate in views:
        for code, cls in ((1, "new development"), (2, "decay")):
            mask = (arr == code) if gate is None else (arr == code) & gate
            OVERLAYS["%s|%s|%s" % (city, view, cls)] = overlay_png(
                mask, dd["extent"], CLASS_COLOR[cls],
                "%s_%s_%s" % (city.lower(), view, cls.replace(" ", "_")))
    print("      %-13s overlays written" % city)

BOUNDS.to_crs("EPSG:4326").to_file(
    os.path.join(WEB, "boundaries.geojson"),
    driver="GeoJSON"
)

segments = []
for city, fips in COUNTY.items():
    part = gpd.read_file(TIGER_ROADS % fips)
    part = part[part["MTFCC"].isin(ROAD_KEEP)].to_crs("EPSG:5070")
    clip = BOUNDS[BOUNDS.city == city].geometry.union_all()
    part = part[part.intersects(clip)].copy()
    part["geometry"] = part.geometry.intersection(clip).simplify(20)
    part = part[~part.geometry.is_empty]
    segments.append(part.assign(city=city)[["city", "MTFCC", "geometry"]])
    print("      %-13s %d road segments" % (city, len(part)))

gpd.GeoDataFrame(pd.concat(segments), crs="EPSG:5070").to_crs("EPSG:4326") \
    .to_file(os.path.join(WEB, "roads.geojson"), driver="GeoJSON")

tracts[["GEOID", "land_km2", "geometry"] + DENSITY].to_crs("EPSG:4326") \
    .to_file(os.path.join(WEB, "tracts.geojson"), driver="GeoJSON")

# Convert NaN -> None for JSON serialization
metrics_json = METRICS.astype(object).where(pd.notna(METRICS), None)

from rasterio.features import shapes as rio_shapes
from shapely.geometry import shape

os.makedirs(os.path.join(WEB, "vectors"), exist_ok=True)

def vector_layer(mask, transform, min_m2=2700, tol_m=15):
    """Polygonize a boolean 30 m mask, drop slivers, simplify in metres."""
    geoms = [shape(g) for g, v in
             rio_shapes(mask.astype(np.uint8), mask=mask, transform=transform)]
    if not geoms:
        return gpd.GeoSeries([], crs="EPSG:5070")
    gs = gpd.GeoSeries(geoms, crs="EPSG:5070")
    gs = gs[gs.area >= min_m2]                 # >= 3 pixels; set 0 to keep all
    gs = gs.simplify(tol_m).buffer(0)          # half-pixel tolerance, fix validity
    return gs[~gs.is_empty]

VECTORS = {}
for city in CITIES:
    dd = DATA[city]
    views = [("detected", PRED[city], None)]
    if city == HOME:
        views += [("observed", dd["label"], dd["holdout"]),
                  ("detectedholdout", PRED[city], dd["holdout"])]
    for view, arr, gate in views:
        for code, cls in ((1, "new development"), (2, "decay")):
            mask = (arr == code) if gate is None else (arr == code) & gate
            gs = vector_layer(mask, dd["transform"])
            name = "%s_%s_%s" % (city.lower(), view, cls.replace(" ", "_"))
            gdf = gpd.GeoDataFrame(
                {"city": city, "view": view, "cls": cls, "area_km2": gs.area / 1e6},
                geometry=gs.values, crs="EPSG:5070").to_crs("EPSG:4326")
            path = os.path.join(WEB, "vectors", name + ".geojson")
            gdf.to_file(path, driver="GeoJSON", COORDINATE_PRECISION=5)
            VECTORS["%s|%s|%s" % (city, view, cls)] = {
                "file": "data/vectors/%s.geojson" % name,
                "color": CLASS_COLOR[cls],
                "features": len(gdf)}
    print("      %-13s vectors written" % city)


with open(os.path.join(WEB, "stats.json"), "w") as handle:
    json.dump({
        "cities": list(CITIES),
        "densityLayers": DENSITY,
        "vectors": VECTORS,
        "prCurves": PR_CURVES,
        "metrics": metrics_json.to_dict("records"),
        "trajectory": TRAJECTORY.to_dict("records"),
        "validation": VALIDATION.to_dict("records"),
        "regime": REGIME.reset_index().to_dict("records"),
        "tracts": tracts[["GEOID"] + DENSITY].to_dict("records")
    }, handle, allow_nan=False)

print("      %d overlays, %d tracts, %d PR curves -> %s"
      % (len(OVERLAYS), len(tracts), len(PR_CURVES), WEB))

      Philadelphia  overlays written
      Detroit       overlays written
      Atlanta       overlays written
      Philadelphia  9453 road segments
      Detroit       7345 road segments
      Atlanta       6436 road segments
      Philadelphia  vectors written
      Detroit       vectors written
      Atlanta       vectors written
      10 overlays, 408 tracts, 6 PR curves -> web/data


In [23]:
from sklearn.cluster import DBSCAN

TIGER_BG = "https://www2.census.gov/geo/tiger/TIGER2024/BG/tl_2024_%s_bg.zip"
ACS = "https://api.census.gov/data/2023/acs/acs5"
ACS_KEY = os.environ.get("a9e713a06a0a0f8ec8531e047c9d01e7d9f507d9", "")  
ACS_VARS = {
    "B01003_001E": "population",
    "B19013_001E": "median hh income",
    "B25077_001E": "median home value",
    "B25002_001E": "housing units", "B25002_003E": "vacant units",
    "B25035_001E": "median year built",
    "B23025_003E": "labor force", "B23025_005E": "unemployed",
    "B15003_001E": "adults 25+", "B15003_022E": "ba", "B15003_023E": "ma",
    "B15003_024E": "prof", "B15003_025E": "phd",
    "C17002_001E": "poverty denom", "C17002_002E": "pov_a", "C17002_003E": "pov_b",
}

def build_acs():
    url = ACS + "?get=" + ",".join(ACS_VARS) \
        + "&for=block%20group:*" \
        + "&in=state:" + COUNTY[HOME][:2] + "%20county:" + COUNTY[HOME][2:] \
        + (("&key=" + "a9e713a06a0a0f8ec8531e047c9d01e7d9f507d9") if "a9e713a06a0a0f8ec8531e047c9d01e7d9f507d9" else "")
    resp = requests.get(url, timeout=300)
    if resp.status_code != 200 or not resp.text.lstrip().startswith("["):
        raise SystemExit("ACS HTTP %d\n%s" % (resp.status_code, resp.text[:400]))
    raw = resp.json()
    df = pd.DataFrame(raw[1:], columns=raw[0]).rename(columns=ACS_VARS)
    df["GEOID"] = df["state"] + df["county"] + df["tract"] + df["block group"]
    num = df[list(ACS_VARS.values())].apply(pd.to_numeric, errors="coerce")
    num = num.mask(num <= -666666666)
    num["vacancy rate"] = num["vacant units"] / num["housing units"].clip(lower=1)
    num["unemployment rate"] = num["unemployed"] / num["labor force"].clip(lower=1)
    num["share bachelors+"] = (num[["ba", "ma", "prof", "phd"]].sum(1)
                               / num["adults 25+"].clip(lower=1))
    num["poverty rate"] = (num[["pov_a", "pov_b"]].sum(1)
                           / num["poverty denom"].clip(lower=1))
    keep = ["population", "median hh income", "median home value",
            "median year built", "vacancy rate", "unemployment rate",
            "share bachelors+", "poverty rate"]
    return {"GEOID": df["GEOID"].to_numpy(), **{k: num[k].to_numpy() for k in keep}}

acs_raw = cached("acs_bg", build_acs)
ACS_COLS = [k for k in acs_raw if k != "GEOID"]
acs = pd.DataFrame({k: acs_raw[k] for k in acs_raw})

# block group geometry, decay aggregation
bg = gpd.read_file(TIGER_BG % COUNTY[HOME][:2]).to_crs("EPSG:5070")
bg = bg[bg["GEOID"].str[:5] == COUNTY[HOME]].merge(acs, on="GEOID")
d = DATA[HOME]
bg_index = rasterize([(g, i + 1) for i, g in enumerate(bg.geometry)],
                     out_shape=d["inside"].shape, transform=d["transform"],
                     fill=0, dtype="int32")
flat, n = bg_index.ravel(), len(bg) + 1
bg["land_km2"] = np.bincount(flat, weights=d["inside"].ravel().astype(float),
                             minlength=n)[1:] * 0.0009
bg["built_km2"] = np.bincount(flat, weights=d["developed"].ravel().astype(float),
                              minlength=n)[1:] * 0.0009
bg["decay_km2"] = np.bincount(flat, weights=(PRED[HOME] == 2).ravel().astype(float),
                              minlength=n)[1:] * 0.0009
bg = bg[bg["land_km2"] > 0.02].copy()
bg["decay density"] = bg["decay_km2"] / bg["land_km2"]
print("      %d block groups joined" % len(bg))

# clustering decay pixels into named hotspots
rr, cc = np.where((PRED[HOME] == 2) & d["inside"])
minx, _, _, maxy = GRID[HOME]["bounds"]
xy = np.column_stack([minx + cc * PIXEL_M, maxy - rr * PIXEL_M])
db = DBSCAN(eps=3 * PIXEL_M, min_samples=25).fit(xy)
sizes = pd.Series(db.labels_[db.labels_ >= 0]).value_counts()
TOP = sizes.head(4).index.tolist()
hotspots = []
for rank, cl in enumerate(TOP, 1):
    pts = xy[db.labels_ == cl]
    cx, cy = pts.mean(0)
    hit = bg[bg.contains(gpd.points_from_xy([cx], [cy])[0])]
    hotspots.append({"rank": rank, "pixels": int(sizes[cl]),
                     "km2": sizes[cl] * 0.0009, "x": cx, "y": cy,
                     "tract": hit["GEOID"].iloc[0][5:11] if len(hit) else "n/a"})
    print("      hotspot %d  %.2f km2  tract %s" % (rank, sizes[cl] * 0.0009,
                                                    hotspots[-1]["tract"]))
HOTSPOTS = pd.DataFrame(hotspots)
HOTSPOTS.to_csv("outputs/decay_hotspots.csv", index=False)

# zoom panels, decay over a socioeconomic choropleth
ZOOM_M = 2200
for var in ["median hh income", "vacancy rate", "poverty rate"]:
    fig, axes = plt.subplots(1, len(TOP), figsize=(4.6 * len(TOP), 6.6))
    values = bg[var].to_numpy()
    ok = np.isfinite(values)
    cuts = np.quantile(values[ok], [.2, .4, .6, .8])
    edges = np.concatenate([[np.nanmin(values) - 1], cuts, [np.nanmax(values) + 1]])
    for ax, spot in zip(np.atleast_1d(axes), hotspots):
        x0, x1 = spot["x"] - ZOOM_M, spot["x"] + ZOOM_M
        y0, y1 = spot["y"] - ZOOM_M, spot["y"] + ZOOM_M
        bg.plot(ax=ax, column=var, cmap=ListedColormap(SEQUENTIAL),
                norm=BoundaryNorm(edges, 5), edgecolor=MUTE, linewidth=0.4,
                missing_kwds={"color": PAPER})
        ax.imshow(np.where(PRED[HOME] == 2, 1.0, np.nan), extent=d["extent"],
                  cmap=ListedColormap([CLASS_COLOR["decay"]]),
                  interpolation="nearest", zorder=4)
        bg.boundary.plot(ax=ax, color=MUTE, linewidth=0.4, zorder=3)
        ax.set_xlim(x0, x1)
        ax.set_ylim(y0, y1)
        ax.set_axis_off()
        ax.set_title("HOTSPOT %d, TRACT %s" % (spot["rank"], spot["tract"]),
                     loc="left", fontsize=12, fontweight="bold", pad=8)
    legend(fig, RAMP_LEGEND + [Patch(facecolor=CLASS_COLOR["decay"], edgecolor=INK,
                                     linewidth=1.6, label="DETECTED DECAY")],
           6, y=0.07, size=9, loc="lower left")
    frame(fig, "DECAY HOTSPOTS OVER %s" % var.upper(),
          "fig11_hotspot_%s.png" % var.replace(" ", "_"), top=0.87, bottom=0.14)

# correlation table, decay density vs block group characteristics
rows = []
for var in ACS_COLS:
    v = bg[var].to_numpy()
    ok = np.isfinite(v) & (bg["population"].to_numpy() > 50)
    rho, p = spearmanr(bg["decay density"].to_numpy()[ok], v[ok])
    control = rankdata(bg["built_km2"].to_numpy()[ok] / bg["land_km2"].to_numpy()[ok])
    rx = rankdata(bg["decay density"].to_numpy()[ok])
    ry = rankdata(v[ok])
    prho, pp = pearsonr(rx - np.polyval(np.polyfit(control, rx, 1), control),
                        ry - np.polyval(np.polyfit(control, ry, 1), control))
    rows.append({"variable": var, "n": int(ok.sum()), "spearman": rho, "p": p,
                 "partial_spearman": prho, "partial_p": pp})
    print("      %-20s rho %+.3f  partial %+.3f" % (var, rho, prho))
BG_CORR = pd.DataFrame(rows).sort_values("spearman")
BG_CORR.to_csv("outputs/bg_correlations.csv", index=False)

bg[["GEOID", "land_km2", "decay density", "geometry"] + ACS_COLS] \
    .to_crs("EPSG:4326").to_file(os.path.join(WEB, "blockgroups.geojson"),
                                 driver="GeoJSON")

      1337 block groups joined
      hotspot 1  0.47 km2  tract 035702
      hotspot 2  0.43 km2  tract 980906
      hotspot 3  0.31 km2  tract 980901
      hotspot 4  0.25 km2  tract 980901
      figures/fig11_hotspot_median_hh_income.png
      figures/fig11_hotspot_vacancy_rate.png
      figures/fig11_hotspot_poverty_rate.png
      population           rho +0.038  partial -0.011
      median hh income     rho +0.086  partial +0.031
      median home value    rho +0.051  partial +0.008
      median year built    rho +0.140  partial +0.051
      vacancy rate         rho -0.070  partial -0.016
      unemployment rate    rho -0.005  partial +0.017
      share bachelors+     rho +0.062  partial +0.014
      poverty rate         rho -0.069  partial -0.006
